In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1. Load FULL dataset
df = pd.read_csv("../../data/df1.csv", low_memory=False)

# Keep a working copy for analysis (subset)
thickness_cols = ["lh_MeanThickness_thickness", "rh_MeanThickness_thickness"]
group_col = "CONCOHORT"
id_col = "PATNO"

df_work = df[[group_col, id_col] + thickness_cols].copy()

In [ ]:
# 2. Map cohort labels
cohort_map = {
    1.0: "PD",
    4.0: "Prodromal PD",
    2.0: "HC"
}
df_work[group_col] = df_work[group_col].map(cohort_map)

In [ ]:
# 3. Basic cleaning
df_work = df_work.dropna(subset=[group_col] + thickness_cols)

for col in thickness_cols:
    df_work = df_work[df_work[col] >= 0]

print("Group sizes BEFORE cleaning:")
print(df_work[group_col].value_counts())

In [ ]:
# 4. Plot (optional but useful)
group_order = ["HC", "PD", "Prodromal PD"]

for col in thickness_cols:
    plt.figure(figsize=(9, 6))
    
    data_to_plot = [df_work[df_work[group_col] == g][col].values for g in group_order]
    plt.boxplot(data_to_plot, tick_labels=group_order)
    
    for i, g in enumerate(group_order, start=1):
        y = df_work[df_work[group_col] == g][col].values
        x = np.random.normal(i, 0.04, size=len(y))
        plt.scatter(x, y, alpha=0.3, s=10)
    
    plt.title(f"{col} by Cohort")
    plt.ylabel("Thickness")
    plt.show()

In [ ]:
# 5. Detect HC lower-bound outliers
hc_df = df_work[df_work[group_col] == "HC"].copy()

hc_outlier_ids = set()

for col in thickness_cols:
    Q1 = hc_df[col].quantile(0.25)
    Q3 = hc_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    
    ids = hc_df.loc[hc_df[col] < lower, id_col]
    
    print(f"\n{col}")
    print(f"Lower bound: {lower:.4f}")
    print(f"Outliers: {ids.tolist()}")
    
    hc_outlier_ids.update(ids.tolist())

hc_outlier_ids = sorted(hc_outlier_ids)

print("\nFinal HC outliers (union across hemispheres):")
print(hc_outlier_ids)
print("Total HC individuals removed:", len(hc_outlier_ids))


In [ ]:
# 6. Remove from FULL dataset
df_clean = df[~df[id_col].isin(hc_outlier_ids)].copy()

In [ ]:
# 7. Sanity check
print("\nGroup sizes AFTER cleaning:")
print(df_clean[group_col].map(cohort_map).value_counts())

In [ ]:
# 8. Save cleaned dataset
df_clean.to_csv("../../data/df1.csv", index=False)

print("\nSaved as df1.csv")